In [ ]:
"""
AS/400 Phase 1 – Raw Extraktion
================================
Zieht unveränderte Rohdaten aus den qsys2-Systemkatalogen.
Keine Berechnungen, keine Joins – nur SELECT * (bzw. alle Spalten).
Ergebnis: eine Excel-Datei mit einem Sheet pro Systemkatalog-View.

Voraussetzungen:
    pip install pyodbc pandas openpyxl

Nutzung:
    1. DSN_NAME (und optional USER/PASSWORD) anpassen
    2. Optional: SCHEMA_FILTER befüllen
    3. python as400_phase1_raw_extract.py
"""

import pyodbc
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
import sys

# ─────────────────────────────────────────────
# KONFIGURATION – hier anpassen
# ─────────────────────────────────────────────
DSN_NAME      = "DEIN_DSN_NAME"
USER          = ""
PASSWORD      = ""

# Leere Liste = alle Schemas (außer System-Schemas).
# Oder z.B. ["MEINLIB", "PRODLIB"]
SCHEMA_FILTER = []

OUTPUT_FILE   = f"as400_raw_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
# ─────────────────────────────────────────────


def connect():
    conn_str = f"DSN={DSN_NAME}"
    if USER:
        conn_str += f";UID={USER};PWD={PASSWORD}"
    try:
        conn = pyodbc.connect(conn_str, timeout=30)
        print(f"✓ Verbunden mit: {DSN_NAME}")
        return conn
    except Exception as e:
        print(f"✗ Verbindungsfehler: {e}")
        sys.exit(1)


def schema_filter_clause(col):
    """WHERE-Klausel für Schema-Filter."""
    system_schemas = "'QSYS','QSYS2','SYSIBM','SYSPROC','SYSTOOLS','QTEMP'"
    if not SCHEMA_FILTER:
        return f"{col} NOT IN ({system_schemas})"
    vals = ", ".join(f"'{s}'" for s in SCHEMA_FILTER)
    return f"{col} IN ({vals})"


# ─────────────────────────────────────────────
# RAW QUERIES – so wenig wie möglich eingreifen.
# Nur WHERE-Klausel zum Schema-Filter, kein JOIN,
# keine Berechnungen, keine CASE-Ausdrücke.
# ─────────────────────────────────────────────
def build_queries():
    return {

        "raw_systables": f"""
            SELECT *
            FROM qsys2.systables
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME
        """,

        "raw_syscolumns": f"""
            SELECT *
            FROM qsys2.syscolumns
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME, ORDINAL_POSITION
        """,

        "raw_systablestat": f"""
            SELECT *
            FROM qsys2.systablestat
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME
        """,

        "raw_syscolumnstat": f"""
            SELECT *
            FROM qsys2.syscolumnstat
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME
        """,

        "raw_sysviews": f"""
            SELECT *
            FROM qsys2.sysviews
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME
        """,

        "raw_sysindexes": f"""
            SELECT *
            FROM qsys2.sysindexes
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME, INDEX_NAME
        """,

        "raw_syskeys": f"""
            SELECT *
            FROM qsys2.syskeys
            WHERE {schema_filter_clause('TABLE_SCHEMA')}
            ORDER BY TABLE_SCHEMA, TABLE_NAME, INDEX_NAME, ORDINAL_POSITION
        """,

        "raw_sysroutines": f"""
            SELECT *
            FROM qsys2.sysroutines
            WHERE {schema_filter_clause('ROUTINE_SCHEMA')}
            ORDER BY ROUTINE_SCHEMA, ROUTINE_NAME
        """,
    }


def run_query(conn, name, sql):
    print(f"  → {name} ...", end=" ", flush=True)
    try:
        df = pd.read_sql(sql, conn)
        print(f"✓  {len(df):>7,} Zeilen  |  {len(df.columns)} Spalten")
        return df, None
    except Exception as e:
        msg = str(e)
        print(f"⚠  Fehler: {msg}")
        return pd.DataFrame({"__fehler__": [msg]}), msg


def style_header(ws, n_cols, tab_color):
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    alt_fill    = PatternFill("solid", fgColor="EBF3FB")
    data_font   = Font(name="Arial", size=10)
    thin        = Side(style="thin", color="D0D0D0")
    border      = Border(left=thin, right=thin, top=thin, bottom=thin)
    center      = Alignment(horizontal="center", vertical="center")
    left        = Alignment(horizontal="left",   vertical="center")

    ws.sheet_properties.tabColor = tab_color
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 20

    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font      = header_font
        cell.fill      = header_fill
        cell.border    = border
        cell.alignment = center

    for row_idx in range(2, ws.max_row + 1):
        fill = alt_fill if row_idx % 2 == 0 else None
        for col_idx in range(1, n_cols + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.font      = data_font
            cell.border    = border
            cell.alignment = left
            if fill:
                cell.fill = fill

    for col_idx in range(1, n_cols + 1):
        col_letter = get_column_letter(col_idx)
        col_cells  = [ws.cell(row=r, column=col_idx).value for r in range(1, min(50, ws.max_row) + 1)]
        max_len    = max((len(str(v)) if v is not None else 0 for v in col_cells), default=8)
        ws.column_dimensions[col_letter].width = min(max_len + 4, 45)


TAB_COLORS = ["2E75B6","00B050","7030A0","FF8C00","C00000","4BACC6","F79646","0070C0","70AD47"]


def write_summary(wb, results, errors, meta):
    ws = wb.create_sheet("00_Info", 0)
    ws.sheet_properties.tabColor = "1F4E79"

    title_font  = Font(name="Arial", bold=True, size=14, color="1F4E79")
    label_font  = Font(name="Arial", bold=True, size=10)
    data_font   = Font(name="Arial", size=10)
    head_fill   = PatternFill("solid", fgColor="1F4E79")
    head_font   = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    ok_font     = Font(name="Arial", size=10, color="00B050")
    err_font    = Font(name="Arial", size=10, color="C00000")

    ws["A1"] = "AS/400 Systemkatalog – Raw Extraktion"
    ws["A1"].font = title_font
    ws["A3"] = "Extraktionsdatum:"
    ws["A3"].font = label_font
    ws["B3"] = meta["datum"]
    ws["B3"].font = data_font
    ws["A4"] = "DSN:"
    ws["A4"].font = label_font
    ws["B4"] = meta["dsn"]
    ws["B4"].font = data_font

    ws["A6"] = "SHEET"
    ws["B6"] = "QSYS2-VIEW"
    ws["C6"] = "ZEILEN"
    ws["D6"] = "SPALTEN"
    ws["E6"] = "STATUS"
    for col in ["A6","B6","C6","D6","E6"]:
        ws[col].font = head_font
        ws[col].fill = head_fill
        ws[col].alignment = Alignment(horizontal="center")

    view_names = {
        "raw_systables":    "qsys2.systables",
        "raw_syscolumns":   "qsys2.syscolumns",
        "raw_systablestat": "qsys2.systablestat",
        "raw_syscolumnstat":"qsys2.syscolumnstat",
        "raw_sysviews":     "qsys2.sysviews",
        "raw_sysindexes":   "qsys2.sysindexes",
        "raw_syskeys":      "qsys2.syskeys",
        "raw_sysroutines":  "qsys2.sysroutines",
    }

    for r, (sheet, df) in enumerate(results.items(), 7):
        err = errors.get(sheet)
        ws.cell(r, 1, sheet).font      = data_font
        ws.cell(r, 2, view_names.get(sheet, sheet)).font = data_font
        ws.cell(r, 3, len(df)).font    = data_font
        ws.cell(r, 3).alignment        = Alignment(horizontal="right")
        ws.cell(r, 4, len(df.columns)).font = data_font
        ws.cell(r, 4).alignment        = Alignment(horizontal="right")
        status_cell = ws.cell(r, 5, "OK" if not err else f"FEHLER: {err[:60]}")
        status_cell.font = ok_font if not err else err_font

    ws["A6"].alignment = Alignment(horizontal="left")
    ws.column_dimensions["A"].width = 22
    ws.column_dimensions["B"].width = 28
    ws.column_dimensions["C"].width = 12
    ws.column_dimensions["D"].width = 10
    ws.column_dimensions["E"].width = 55

    ws["A9+2"] if False else None  # placeholder
    note_row = len(results) + 9
    ws.cell(note_row, 1, "Nächster Schritt:").font = Font(name="Arial", bold=True, size=10)
    ws.cell(note_row+1, 1, "→ as400_phase1_transform.py ausführen").font = Font(name="Arial", size=10, color="2E75B6")


def main():
    print("\n=== AS/400 Phase 1 – Raw Extraktion ===\n")

    conn    = connect()
    queries = build_queries()

    results = {}
    errors  = {}
    for name, sql in queries.items():
        df, err = run_query(conn, name, sql)
        results[name] = df
        if err:
            errors[name] = err

    conn.close()
    print(f"\n✓ Verbindung geschlossen")
    print(f"\nSchreibe: {OUTPUT_FILE} ...")

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        for name, df in results.items():
            df.to_excel(writer, sheet_name=name, index=False)

    wb = load_workbook(OUTPUT_FILE)
    for i, ws_name in enumerate(wb.sheetnames):
        ws = wb[ws_name]
        style_header(ws, len(results[ws_name].columns), TAB_COLORS[i % len(TAB_COLORS)])

    write_summary(wb, results, errors, {
        "datum": datetime.now().strftime("%d.%m.%Y %H:%M"),
        "dsn":   DSN_NAME,
    })

    wb.save(OUTPUT_FILE)

    print(f"✓ Fertig: {OUTPUT_FILE}")
    if errors:
        print(f"\n⚠ {len(errors)} View(s) konnten nicht abgefragt werden:")
        for name, msg in errors.items():
            print(f"   {name}: {msg}")
        print("→ Bitte Ansichten prüfen und ggf. SCHEMA_FILTER oder Query anpassen.")
    print(f"\nNächster Schritt: python as400_phase1_transform.py {OUTPUT_FILE}\n")


if __name__ == "__main__":
    main()